# American Option Pricing with a Binomial Tree (Cox-Ross-Rubinstein)

This notebook prices **American and European Call and Put options** on **KO (Coca-Cola)** with live data from `yfinance`, using the **CRR binomial tree**.

**American vs European:**
- **European** option: can only be exercised **at expiry** - Black-Scholes gives an exact formula for it.
- **American** option: can be exercised **any time up to expiry** (this is how most stock options work). Because of this early-exercise right there is *no simple formula* - so we build a **binomial tree**.

**Plan:**
1. fetch KO data + risk-free rate
2. historical volatility from log returns
3. tree parameters ($u$, $d$, $p$) and the pricing function
4. price American & European options + early-exercise premium
5. convergence check vs the Black-Scholes formula
6. implied volatility with Brent's method

## Inputs

| Input | Symbol | Where we get it |
|---|---|---|
| Current stock price | $S_0$ | Market (yfinance, latest close) |
| Strike price | $K$ | Option contract |
| Time to maturity | $T$ | Expiry date - today (in years) |
| Risk-free rate | $r$ | Treasury bill yield (`^IRX`) |
| Volatility | $\sigma$ | Historical log returns of the stock |
| **Tree steps** | $N$ | Chosen by us (more steps = more accurate, slower) |

All three notebooks of this project price on **KO**, so their results can be compared. This first notebook assumes **no dividends**.

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime
import scipy.stats as stats
from scipy.optimize import brentq

# Fetch 1 year of historical daily data for KO
ticker_symbol = "KO"
df = yf.Ticker(ticker_symbol).history(period="1y")

# Extract the most recent closing price as the current stock price (S0)
S0 = df['Close'].iloc[-1]
print(f"Current {ticker_symbol} Stock Price (S0): ${S0:.2f}")

Current KO Stock Price (S0): $93.80


## Volatility from historical data

The model needs $\sigma$ = **annualized volatility** of the stock's log returns.

Daily log return:

$$r_t = \ln\!\left(\frac{P_t}{P_{t-1}}\right)$$

Log returns add up over time, so their standard deviation grows with the square root of time. With 252 trading days in a year:

$$\boxed{\sigma_{\text{annual}} = \operatorname{std}(r_t)\times\sqrt{252}}$$

**Assumption:** log returns are (roughly) normally distributed with constant volatility - this is what the binomial tree also assumes when it picks the up/down factors $u$ and $d$.

In [2]:
# Calculate daily log returns: ln(P_t / P_t-1)
df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))

# Drop missing values if any
log_returns = df['Log_Return'].dropna()

# standard deviation and annualization
daily_volatility = log_returns.std()
historical_volatility = daily_volatility * np.sqrt(252)

print(f"Daily Volatility: {daily_volatility:.6f}")
print(f"Annualized Historical Volatility (sigma): {historical_volatility:.4f} ({historical_volatility * 100:.2f}%)")

Daily Volatility: 0.011935
Annualized Historical Volatility (sigma): 0.1895 (18.95%)


## How the binomial tree works (Cox-Ross-Rubinstein)

**Idea:** cut the option's life into $N$ small steps of length $\Delta t = T/N$. At every step the stock either moves **up** or **down** by a fixed factor:

$$\Delta t = \frac{T}{N}, \qquad u = e^{\sigma\sqrt{\Delta t}}, \qquad d = \frac{1}{u} = e^{-\sigma\sqrt{\Delta t}}$$

(the factors are chosen so the stock's volatility over one step is exactly $\sigma$).

**Risk-neutral probability of an up move:**

$$p = \frac{e^{r\,\Delta t} - d}{u - d}$$

It is the probability that makes the stock's expected return equal to the risk-free rate - no real-world forecasting needed (the same trick Black-Scholes uses).

**Pricing = backward induction:**
1. at expiry compute the payoff: call $\max(S_T - K,\,0)$, put $\max(K - S_T,\,0)$
2. step backwards one node at a time:

$$\text{value} = e^{-r\,\Delta t}\Big[p\cdot\text{value}_{up} + (1-p)\cdot\text{value}_{down}\Big]$$

3. **American rule** - at every node the holder compares

$$\text{value} = \max\Big(\text{exercise payoff},\ \text{continuation value}\Big)$$

European options skip step 3 (they can only exercise at expiry).

**Good to know:** as $N \to \infty$ the tree converges to the Black-Scholes price - we check this below.

## Definition of option parameters

- **Strike** $K$ - nearest-dollar strike (at the money)
- **Time to maturity** $T = \text{days to expiry}/365.25$
- **Risk-free rate** $r$ - from `^IRX`
- **Volatility** $\sigma$ - from KO's historical log returns
- **Steps** $N = 500$ - number of tree steps (accuracy vs speed)

The cell also builds the tree parameters $u$, $d$, $p$ and defines `crr_price()` - the function that runs the backward induction.

In [3]:
# Define option parameters

K = round(S0)                # nearest-dollar strike = at the money
expiry_date = datetime(2027, 6, 18)
today = datetime.now()

# ----------------------------------------------------------------------- #

# time to maturity (T) in years
T = (expiry_date - today).days / 365.25
print(f"Strike Price (K)           : ${K}")
print(f"Time to Maturity (T)       : {T:.8f} years ({(expiry_date - today).days} days)")

# Risk-Free Rate from ^IRX and historical volatility:
irx = yf.Ticker("^IRX")
# recent close price
r = irx.history(period="5d")['Close'].iloc[-1] / 100
sigma = historical_volatility
print(f"Risk-Free Rate (r)         : {r:.4f} ({r * 100:.2f}%)")
print(f"Volatility Used (sigma)    : {sigma:.4f} ({sigma * 100:.2f}%)")

# binomial tree parameters (Cox-Ross-Rubinstein)
N = 500                      # number of steps
dt = T / N
u = np.exp(sigma * np.sqrt(dt))
d = 1.0 / u
p = (np.exp(r * dt) - d) / (u - d)
print(f"Tree Steps (N)             : {N}    dt = {dt:.6f}")
print(f"Up (u) / Down (d) / Prob (p): {u:.6f} / {d:.6f} / {p:.4f}")

def crr_price(S, K, T, r, sigma, N, option='call', style='american'):
    """
    Price an option with the Cox-Ross-Rubinstein binomial tree.
    option = 'call' or 'put'
    style  = 'american' : exercise allowed at every node (max of exercise vs continuation)
             'european' : exercise only at expiry
    """
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1.0 / u
    p = (np.exp(r * dt) - d) / (u - d)
    disc = np.exp(-r * dt)

    # option value at expiry (last step): stock prices S*u^(N-j)*d^j
    ST = S * u ** (np.arange(N, -1, -1)) * d ** (np.arange(0, N + 1))
    if option == 'call':
        vals = np.maximum(ST - K, 0.0)
    else:
        vals = np.maximum(K - ST, 0.0)

    # step backwards through the tree
    for i in range(N - 1, -1, -1):
        Si = S * u ** (np.arange(i, -1, -1)) * d ** (np.arange(0, i + 1))
        vals = disc * (p * vals[:-1] + (1 - p) * vals[1:])
        if style == 'american':
            if option == 'call':
                vals = np.maximum(vals, Si - K)
            else:
                vals = np.maximum(vals, K - Si)
    return float(vals[0])

Strike Price (K)           : $94
Time to Maturity (T)       : 0.65982204 years (241 days)
Risk-Free Rate (r)         : 0.0408 (4.08%)
Volatility Used (sigma)    : 0.1895 (18.95%)
Tree Steps (N)             : 500    dt = 0.001320
Up (u) / Down (d) / Prob (p): 1.006906 / 0.993141 / 0.5022


## Option prices and the early-exercise premium

The cell below prices all four combinations (American/European $\times$ call/put).

**Early-exercise premium:**

$$\text{premium} = \text{American price} - \text{European price} \;\ge\ 0$$

What to expect with **no dividends**:
- **American put > European put** - selling early is sometimes valuable (you collect the strike sooner).
- **American call = European call** - exercising early means giving up the option's remaining time value and paying the strike early (you even lose the interest on it), for no benefit. So the tree never picks early exercise for the call and the premium is exactly 0.

In [4]:
# option prices
call_amer = crr_price(S0, K, T, r, sigma, N, 'call', 'american')
put_amer = crr_price(S0, K, T, r, sigma, N, 'put', 'american')
call_eur = crr_price(S0, K, T, r, sigma, N, 'call', 'european')
put_eur = crr_price(S0, K, T, r, sigma, N, 'put', 'european')

print("=" * 45)
print(f"Underlying Asset   : {ticker_symbol}")
print(f"Current Stock Price: ${S0:.2f}")
print(f"Strike Price       : ${K}")
print(f"Expiration Date    : {expiry_date}    T = {T:.4f}")
print(f"Volatility Used    : {sigma * 100:.2f}%")
print(f"Tree Steps (N)     : {N}")
print("-" * 45)
print(f"American Call      : ${call_amer:.2f}")
print(f"American Put       : ${put_amer:.2f}")
print(f"European Call      : ${call_eur:.2f}")
print(f"European Put       : ${put_eur:.2f}")
print("-" * 45)
print(f"Early-exercise premium  Call: ${call_amer - call_eur:.4f}   (expect 0 without dividends)")
print(f"Early-exercise premium  Put : ${put_amer - put_eur:.4f}   (expect > 0)")
print("=" * 45)

Underlying Asset   : KO
Current Stock Price: $93.80
Strike Price       : $94
Expiration Date    : 2027-06-18 00:00:00    T = 0.6598
Volatility Used    : 18.95%
Tree Steps (N)     : 500
---------------------------------------------
American Call      : $6.90
American Put       : $4.86
European Call      : $6.90
European Put       : $4.61
---------------------------------------------
Early-exercise premium  Call: $0.0000   (expect 0 without dividends)
Early-exercise premium  Put : $0.2469   (expect > 0)


## Does the tree converge?

Two accuracy checks:

1. **More steps = better price.** The tree's error shrinks roughly like $O(1/N)$: doubling $N$ roughly halves the error. The table below shows the prices for $N$ from 50 up to 1000.
2. **European prices must match Black-Scholes.** A European option priced by a big tree must be almost identical to the closed-form Black-Scholes price - a free cross-check of both implementations:

$$C = S_0 N(d_1) - K e^{-rT} N(d_2), \qquad d_1 = \frac{\ln(S_0/K) + (r + \tfrac12\sigma^2)T}{\sigma\sqrt{T}},\quad d_2 = d_1 - \sigma\sqrt{T}$$

In [5]:
# Black-Scholes closed form (validation reference)
def bs_european(S0, K, T, r, sigma, option='call'):
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option == 'call':
        return S0 * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)
    return K * np.exp(-r * T) * stats.norm.cdf(-d2) - S0 * stats.norm.cdf(-d1)

bs_call = bs_european(S0, K, T, r, sigma, 'call')
bs_put = bs_european(S0, K, T, r, sigma, 'put')

print(f"Black-Scholes (exact) : Call ${bs_call:.4f}   Put ${bs_put:.4f}")
print(f"Tree at N = {N:<6}    : Call ${call_eur:.4f}   Put ${put_eur:.4f}")
print(f"                        diff {abs(call_eur - bs_call):.6f}      {abs(put_eur - bs_put):.6f}\n")

print(f"{'N':>6} {'Amer call':>12} {'Eur call':>12} {'Eur put':>12} {'Amer put':>12}")
for N_test in [50, 100, 250, 500, 1000]:
    ac = crr_price(S0, K, T, r, sigma, N_test, 'call', 'american')
    ec = crr_price(S0, K, T, r, sigma, N_test, 'call', 'european')
    ep = crr_price(S0, K, T, r, sigma, N_test, 'put', 'european')
    ap = crr_price(S0, K, T, r, sigma, N_test, 'put', 'american')
    print(f"{N_test:>6} {ac:>12.4f} {ec:>12.4f} {ep:>12.4f} {ap:>12.4f}")

Black-Scholes (exact) : Call $6.9036   Put $4.6086
Tree at N = 500       : Call $6.9037   Put $4.6087
                        diff 0.000044      0.000044

     N    Amer call     Eur call      Eur put     Amer put
    50       6.8853       6.8853       4.5903       4.8494


   100       6.8964       6.8964       4.6014       4.8536
   250       6.9022       6.9022       4.6072       4.8554
   500       6.9037       6.9037       4.6087       4.8556


  1000       6.9041       6.9041       4.6091       4.8555


## Implied Volatility using Brent's root-finding method

**Normal way:** we know $\sigma$ -> compute the price (here: run the tree). **Implied volatility:** we know the *market price* -> find the $\sigma$ that makes the tree produce it:

$$\text{find } \sigma \text{ such that } \text{Tree}(S_0, K, T, r, \sigma) = \text{market price}$$

Same Brent root-finder as in the European project - only the pricer inside is now the binomial tree (a bit slower, same idea). We search American prices, because listed stock options are American.

In [6]:
def tree_price(CP, S0, K, sigma, T, r):
    """CRR binomial price (American style) used by the root-finder."""
    if sigma <= 0:
        return 0.0
    option = 'call' if CP.upper() == 'CALL' else 'put'
    return crr_price(S0, K, T, r, sigma, N, option, 'american')

def implied_volatility(market_price, CP, S0, K, T, r):
    """
    Finds the implied volatility by reverse-engineering the binomial tree price
    using Brent's root-finding method.
    """
    # objective function: tree price - market price
    def objective_function(sigma):
        return tree_price(CP, S0, K, sigma, T, r) - market_price

    try:
        # search for volatility between 1% and 500%
        iv = brentq(objective_function, 1e-2, 5.0)
        return iv
    except ValueError:
        # returns NaN if the market price is invalid
        return np.nan

if __name__ == "__main__":
    market_price_call = 4.50    # observed market prices
    market_price_put = 3.50

    iv_call = implied_volatility(market_price_call, 'CALL', S0, K, T, r)
    iv_put = implied_volatility(market_price_put, 'PUT', S0, K, T, r)

    print()
    print(f"Market prices used : call ${market_price_call:.2f}, put ${market_price_put:.2f}")
    print()
    print(f"Implied Volatility_CALL: {iv_call:.4f} ({iv_call * 100:.2f}%)")
    print()
    print(f"Implied Volatility_PUT: {iv_put:.4f} ({iv_put * 100:.2f}%)")


Market prices used : call $4.50, put $3.50

Implied Volatility_CALL: 0.1074 (10.74%)

Implied Volatility_PUT: 0.1431 (14.31%)
